# Business Entity Resolution 

**Target:** 99th percentile F₀.₅ score
**Current Baseline:** F₀.₅ = 0.772 (holdout)

This notebook walks through the complete pipeline step-by-step. Run each cell in order.

## 0. Environment Setup & Verification

In [1]:
# Check environment
import sys, subprocess, json, os, platform
print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"CPU cores: {os.cpu_count()}")

# Verify key packages
import numpy as np
import lightgbm as lgb
import numba
import polars as pl
import rapidfuzz
print(f"NumPy: {np.__version__}")
print(f"LightGBM: {lgb.__version__}")
print(f"Numba: {numba.__version__}")
print(f"Polars: {pl.__version__}")
print(f"RapidFuzz: {rapidfuzz.__version__}")

# Check GPU (optional)
try:
    import torch
    print(f"CUDA available: {torch.cuda.is_available()}")
except:
    print("PyTorch not installed (not needed)")

Python: 3.13.5 (tags/v3.13.5:6cb20a2, Jun 11 2025, 16:15:46) [MSC v.1943 64 bit (AMD64)]
Platform: Windows-11-10.0.22631-SP0
CPU cores: 8


ModuleNotFoundError: No module named 'rapidfuzz'

In [ ]:
# Verify project structure
BASE = "/home/ubuntu/business-entity-resolution/student_resource"  # Adjust if different
os.chdir(BASE)
print(f"Working dir: {os.getcwd()}")

for d in ["dataset/train", "dataset/test", "code/business_entity_resolution/src", "utils", "output", "work"]:
    path = os.path.join(BASE, d)
    exists = os.path.exists(path)
    size = ""
    if exists and os.path.isdir(path):
        total = sum(os.path.getsize(os.path.join(dp, f)) for dp, dn, fn in os.walk(path) for f in fn)
        size = f" ({total/1e9:.2f} GB)"
    print(f"  {d}: {'✓' if exists else '✗'}{size}")

# Check key files
for f in ["work/model.txt", "work/threshold.npy", "code/business_entity_resolution/requirements.txt"]:
    path = os.path.join(BASE, f)
    print(f"  {f}: {'✓' if os.path.exists(path) else '✗'}")

## 1. Exploratory Data Analysis

In [ ]:
# Load training data samples
import polars as pl

train_s1 = pl.read_csv("dataset/train/train_source1.tsv", separator="\t", n_rows=50000)
train_s2 = pl.read_csv("dataset/train/train_source2.tsv", separator="\t", n_rows=50000)
train_s3 = pl.read_csv("dataset/train/train_source3.tsv", separator="\t", n_rows=50000)
gt = pl.read_csv("dataset/train/train_ground_truth.tsv", separator="\t")

print(f"Train S1: {train_s1.shape}")
print(f"Train S2: {train_s2.shape}")
print(f"Train S3: {train_s3.shape}")
print(f"Ground Truth: {gt.shape}")

In [2]:
# Country distribution
for df, name in [(train_s1, "S1"), (train_s2, "S2"), (train_s3, "S3")]:
    vc = df["country"].value_counts()
    print(f"\n{name} countries:")
    print(vc)

NameError: name 'train_s1' is not defined

In [3]:
# Ground truth analysis
gt_parsed = gt.with_columns(
    pl.col("matched_entity_ids").str.split(",").alias("matches")
).with_columns(
    pl.col("matches").list.lengths().alias("n_matches")
)
print("Match count distribution:")
print(gt_parsed["n_matches"].value_counts().sort("n_matches"))

singleton_rate = (gt_parsed["n_matches"] == 0).mean()
print(f"\nSingleton rate: {singleton_rate:.3f}")

NameError: name 'gt' is not defined

In [4]:
# Name/address length stats
for col in ["business_name", "business_address"]:
    for df, name in [(train_s1, "S1"), (train_s2, "S2"), (train_s3, "S3")]:
        lengths = df[col].str.lengths()
        print(f"{name} {col}: mean={lengths.mean():.1f}, median={lengths.median():.1f}, max={lengths.max()}")

NameError: name 'train_s1' is not defined

In [5]:
# Check for Indic scripts
import re
indic_pattern = re.compile(r"[\u0900-\u0D7F]")

for df, name in [(train_s2, "S2"), (train_s3, "S3")]:
    indic = df.filter(pl.col("business_name").str.contains(r"[\u0900-\u0D7F]"))
    print(f"{name} Indic script names: {indic.height} ({indic.height/len(df)*100:.1f}%)")
    if indic.height > 0:
        print(indic.select(["entity_id", "business_name", "country"]).head(3))

NameError: name 'train_s2' is not defined

In [6]:
# Test set overview
test_s1 = pl.read_csv("dataset/test/test_source1.tsv", separator="\t", n_rows=10000)
test_s2 = pl.read_csv("dataset/test/test_source2.tsv", separator="\t", n_rows=10000)
test_s3 = pl.read_csv("dataset/test/test_source3.tsv", separator="\t", n_rows=10000)

print(f"Test S1 sample: {test_s1.shape}")
print(f"Test S2 sample: {test_s2.shape}")
print(f"Test S3 sample: {test_s3.shape}")

for df, name in [(test_s1, "S1"), (test_s2, "S2"), (test_s3, "S3")]:
    vc = df["country"].value_counts()
    print(f"\nTest {name} countries:")
    print(vc)

Test S1 sample: (10000, 4)
Test S2 sample: (10000, 4)
Test S3 sample: (10000, 4)

Test S1 countries:
shape: (3, 2)
┌─────────┬───────┐
│ country ┆ count │
│ ---     ┆ ---   │
│ str     ┆ u32   │
╞═════════╪═══════╡
│ US      ┆ 3891  │
│ India   ┆ 4619  │
│ France  ┆ 1490  │
└─────────┴───────┘

Test S2 countries:
shape: (3, 2)
┌─────────┬───────┐
│ country ┆ count │
│ ---     ┆ ---   │
│ str     ┆ u32   │
╞═════════╪═══════╡
│ US      ┆ 3802  │
│ France  ┆ 1449  │
│ India   ┆ 4749  │
└─────────┴───────┘

Test S3 countries:
shape: (3, 2)
┌─────────┬───────┐
│ country ┆ count │
│ ---     ┆ ---   │
│ str     ┆ u32   │
╞═════════╪═══════╡
│ US      ┆ 3849  │
│ India   ┆ 4716  │
│ France  ┆ 1435  │
└─────────┴───────┘


## 2. Quick Smoke Test (5 minutes)

Run the full pipeline on a small slice to verify everything works.

In [7]:
# Run smoke test prediction
import subprocess, time

cmd = [
    "python", "code/business_entity_resolution/src/predict_test.py",
    "--model", "work/model.txt",
    "--threshold", "work/threshold.npy",
    "--batch", "1000",
    "--topk", "50",
    "--limit-corpus", "100000",
    "--limit-s1", "1000",
    "--tag", "_smoke",
    "--out-dir", "output_smoke",
    "--workers", "8"
]

print("Running smoke test...")
start = time.time()
result = subprocess.run(cmd, capture_output=True, text=True, cwd=BASE)
elapsed = time.time() - start

print(f"Exit code: {result.returncode}")
print(f"Time: {elapsed:.0f}s")
print("STDOUT:")
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.stderr:
    print("STDERR:")
    print(result.stderr[-2000:])

Running smoke test...


NameError: name 'BASE' is not defined

In [8]:
# Validate smoke test output
cmd = [
    "python", "utils/validate_submission.py",
    "--matching", "output_smoke/matching_results.tsv",
    "--candidate", "output_smoke/candidate_pairs.tsv",
    "--test-dir", "dataset/test"
]

result = subprocess.run(cmd, capture_output=True, text=True, cwd=BASE)
print(f"Exit code: {result.returncode}")
print(result.stdout)
if result.stderr:
    print(result.stderr)

NameError: name 'BASE' is not defined

## 3. Full Test Inference (2-4 hours)

**Only run this after smoke test passes!** This produces the actual submission files.

In [9]:
# FULL INFERENCE - UNCOMMENT TO RUN
# This takes 2-4 hours on r6g.4xlarge

# cmd = [
#     "python", "code/business_entity_resolution/src/predict_test.py",
#     "--model", "work/model.txt",
#     "--threshold", "work/threshold.npy",
#     "--batch", "100000",
#     "--topk", "100",
#     "--key-budget", "14",
#     "--per-key-cap", "80",
#     "--workers", "16",
#     "--tag", "_full",
#     "--out-dir", "output"
# ]
# 
# print("Starting full inference...")
# start = time.time()
# result = subprocess.run(cmd, capture_output=True, text=True, cwd=BASE)
# elapsed = time.time() - start
# 
# print(f"Exit code: {result.returncode}")
# print(f"Time: {elapsed/3600:.1f} hours")
# print(result.stdout[-5000:])
# if result.stderr:
#     print(result.stderr[-3000:])

print("Cell ready - uncomment and run when ready for full inference")

Cell ready - uncomment and run when ready for full inference


## 4. Validate Full Output

In [10]:
# Validate full output (run after full inference completes)
cmd = [
    "python", "utils/validate_submission.py",
    "--matching", "output/matching_results.tsv",
    "--candidate", "output/candidate_pairs.tsv",
    "--test-dir", "dataset/test",
    "--check-ids"
]

result = subprocess.run(cmd, capture_output=True, text=True, cwd=BASE)
print(f"Exit code: {result.returncode}")
print(result.stdout)
if result.stderr:
    print(result.stderr)

NameError: name 'BASE' is not defined

## 5. Model & Pipeline Inspection

In [11]:
# Inspect trained model
import lightgbm as lgb
import numpy as np

model = lgb.Booster(model_file="work/model.txt")
thr = np.load("work/threshold.npy")[0]

print(f"Model trees: {model.num_trees()}")
print(f"Best iteration: {model.best_iteration}")
print(f"Threshold: {thr:.3f}")
print(f"Features: {model.feature_name()}")

# Feature importance
imp = sorted(zip(model.feature_name(), model.feature_importance("gain")), key=lambda x: -x[1])
print("\nTop 20 features by gain:")
for name, gain in imp[:20]:
    print(f"  {name:<25} {gain:,.0f}")

Model trees: 700
Best iteration: -1
Threshold: 0.770
Features: ['tok_shared', 'tok_jaccard', 'tok_shared_name', 'tok_shared_addr', 'tok_shared_r0', 'tok_shared_r1', 'tok_shared_r2', 'tok_sizesum', 'tok_sizeratio', 'tok_rare_jaccard', 'tok_name_jaccard', 'tok_addr_jaccard', 'state_match', 'country_match', 'block_score', 'block_score_log', 'name_ratio', 'name_partial', 'name_toksort', 'name_tokset', 'addr_ratio', 'addr_partial', 'addr_toksort', 'addr_tokset', 'cname_ratio', 'cname_tokset', 'caddr_ratio', 'name_len_ratio', 'addr_len_ratio', 'cname_len_ratio']

Top 20 features by gain:
  tok_jaccard               9,078,115
  name_partial              468,709
  addr_tokset               367,666
  tok_addr_jaccard          316,535
  cname_ratio               280,686
  name_toksort              194,484
  state_match               187,776
  addr_len_ratio            121,890
  name_tokset               109,545
  name_ratio                107,333
  addr_ratio                90,528
  tok_name_jac

In [12]:
# Inspect corpus store
from norm_store import Store
import numpy as np

corpus = Store("work/store/test_corpus_full")
print(f"Corpus records: {corpus.n:,}")
print(f"Countries: {dict(zip(*np.unique(corpus.country, return_counts=True)))}")
print(f"Unique states: {len(np.unique(corpus.state))}")

# Sample records
for i in [0, 1000, 10000, 100000]:
    if i < corpus.n:
        print(f"\nRecord {i}:")
        print(f"  ID: {corpus.id(i)}")
        print(f"  Name: {corpus.name_str(i)[:80]}")
        print(f"  Addr: {corpus.addr_str(i)[:80]}")
        print(f"  Country: {corpus.country[i]}, State hash: {corpus.state[i]}")

ModuleNotFoundError: No module named 'norm_store'

In [13]:
# Inspect inverted index
from blocking import Index

idx = Index.load("work/store/test_full_index", n_parts=8)
print(f"Index parts: {len(idx.parts)}")
print(f"Total postings: {idx.total_postings():,}")
print(f"Avg postings/part: {idx.total_postings()/len(idx.parts):,.0f}")

ModuleNotFoundError: No module named 'blocking'

## 6. Hyperparameter Tuning Experiments

Run controlled experiments to improve F₀.₅. Each cell is an independent experiment.

In [14]:
# Experiment 1: Vary top_k (candidates per entity)
# Smaller top_k = smaller candidate_pairs.tsv = better ranking if recall maintained

def run_experiment(topk, key_budget=14, per_key_cap=80, limit_s1=5000, tag_suffix=""):
    import subprocess, time
    tag = f"_exp_topk{topk}_kb{key_budget}_pk{per_key_cap}{tag_suffix}"
    out_dir = f"output_exp_topk{topk}"
    cmd = [
        "python", "code/business_entity_resolution/src/predict_test.py",
        "--model", "work/model.txt",
        "--threshold", "work/threshold.npy",
        "--batch", "5000",
        "--topk", str(topk),
        "--key-budget", str(key_budget),
        "--per-key-cap", str(per_key_cap),
        "--limit-s1", str(limit_s1),
        "--tag", tag,
        "--out-dir", out_dir,
        "--workers", "8"
    ]
    print(f"Running: topk={topk}, key_budget={key_budget}, per_key_cap={per_key_cap}")
    start = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=BASE)
    elapsed = time.time() - start
    
    # Validate
    val_cmd = [
        "python", "utils/validate_submission.py",
        "--matching", f"{out_dir}/matching_results.tsv",
        "--candidate", f"{out_dir}/candidate_pairs.tsv",
        "--test-dir", "dataset/test"
    ]
    val_result = subprocess.run(val_cmd, capture_output=True, text=True, cwd=BASE)
    
    return {
        "topk": topk, "key_budget": key_budget, "per_key_cap": per_key_cap,
        "time": elapsed, "exit_code": result.returncode,
        "val_exit": val_result.returncode, "val_output": val_result.stdout
    }

# Run grid (comment out to run - takes ~10 min each)
# results = []
# for topk in [60, 80, 100, 120]:
#     for kb in [12, 14, 16]:
#         for pk in [60, 80, 100]:
#             results.append(run_experiment(topk, kb, pk, limit_s1=2000))
# 
# import pandas as pd
# df = pd.DataFrame(results)
# print(df.to_string())

print("Experiment function defined. Uncomment loop to run grid search.")

Experiment function defined. Uncomment loop to run grid search.


In [15]:
# Experiment 2: Threshold sweep on holdout (if you have holdout predictions)
# This requires training data - run collect_train.py + train_model.py first

def threshold_sweep():
    # Load holdout predictions from training
    # This is done in train_model.py automatically
    # But you can re-evaluate with different thresholds here
    pass

# Quick threshold test on existing model
import numpy as np
import lightgbm as lgb
from pipeline import build_corpus_and_index, open_corpus, process_batch
from norm_store import Store
from pair_features import FeatureBuilder

# Load model
model = lgb.Booster(model_file="work/model.txt")

# Test different thresholds on a small slice
for thr in [0.65, 0.70, 0.75, 0.77, 0.80, 0.85, 0.90]:
    print(f"Threshold {thr:.2f}: would need holdout data to evaluate F0.5")

print("\nBest threshold from training: 0.770 (F0.5=0.772)")

ModuleNotFoundError: No module named 'pipeline'

## 7. Advanced: Retraining with Custom Hyperparameters

In [16]:
# RETRAINING PIPELINE - Run these sequentially if you want to retrain

# Step 1: Collect training pairs (takes ~30 min)
# cmd = [
#     "python", "code/business_entity_resolution/src/collect_train.py",
#     "--n-s1", "200000",
#     "--topk", "60",
#     "--batch", "30000",
#     "--workers", "8"
# ]
# subprocess.run(cmd, cwd=BASE)

# Step 2: Train model with custom params (takes ~20 min)
# Edit train_model.py params dict first, then:
# cmd = [
#     "python", "code/business_entity_resolution/src/train_model.py",
#     "--chunks", "work/cand/train_chunks",
#     "--holdout-frac", "0.15",
#     "--max-neg-ratio", "8.0",
#     "--rounds", "1000",
#     "--model-out", "work/model_custom.txt",
#     "--seed", "42"
# ]
# subprocess.run(cmd, cwd=BASE)

print("Retraining commands ready. Modify train_model.py params first, then uncomment.")

Retraining commands ready. Modify train_model.py params first, then uncomment.


## 8. France-Specific Analysis (Test-Only Country)

In [17]:
# Analyze France records in test set
import polars as pl

test_s2 = pl.read_csv("dataset/test/test_source2.tsv", separator="\t")
test_s3 = pl.read_csv("dataset/test/test_source3.tsv", separator="\t")

fr_s2 = test_s2.filter(pl.col("country").str.to_lowercase() == "france")
fr_s3 = test_s3.filter(pl.col("country").str.to_lowercase() == "france")

print(f"France S2: {fr_s2.height:,}")
print(f"France S3: {fr_s3.height:,}")
print(f"Total France corpus: {fr_s2.height + fr_s3.height:,}")

# Sample France records
print("\nSample France S2:")
print(fr_s2.select(["entity_id", "business_name", "business_address"]).head(5))
print("\nSample France S3:")
print(fr_s3.select(["entity_id", "business_name", "business_address"]).head(5))

France S2: 703,378
France S3: 731,615
Total France corpus: 1,434,993

Sample France S2:
shape: (5, 3)
┌──────────────┬─────────────────────────────────┬─────────────────────────────────┐
│ entity_id    ┆ business_name                   ┆ business_address                │
│ ---          ┆ ---                             ┆ ---                             │
│ str          ┆ str                             ┆ str                             │
╞══════════════╪═════════════════════════════════╪═════════════════════════════════╡
│ S2-566025912 ┆ Marina Ecole France Sarl        ┆ 63 R. DE DIEPPE, LILLE, Hauts-… │
│ S2-158121477 ┆ SCI Ptit Àmicale                ┆ 18 RUE JEN ZAY, Dunkerque, Nor… │
│ S2-770341988 ┆ sci ligue ici parents           ┆ NO. 5 ALLÉE DES HÊTRES, Pornic… │
│ S2-647447093 ┆ Tunisie Inter Cie Internationa… ┆ 77 AV LEON JOUHAUX, LILLE       │
│ S2-262646344 ┆ OZT ÀMICALE SAS                 ┆ 24 R DESAIX, TOURCOING, Hauts-… │
└──────────────┴────────────────────────────────

In [18]:
# Check if French regions are in FR_REGIONS (normalize.py)
from normalize import FR_REGIONS
print(f"FR_REGIONS ({len(FR_REGIONS)}):")
for r in sorted(FR_REGIONS):
    print(f"  {r}")

ModuleNotFoundError: No module named 'normalize'

## 9. Output Analysis & Error Investigation

In [ ]:
# Analyze matching_results.tsv (after full inference)
import polars as pl

# Only run if output exists
matching_path = "output/matching_results.tsv"
if os.path.exists(matching_path):
    matches = pl.read_csv(matching_path, separator="\t")
    print(f"Total S1 entities: {matches.height:,}")
    
    # Count matches per entity
    matches = matches.with_columns(
        pl.col("matched_entity_ids").str.split(",").alias("match_list")
    ).with_columns(
        pl.when(pl.col("matched_entity_ids") == "").then(0)
        .otherwise(pl.col("match_list").list.lengths()).alias("n_matches")
    )
    
    print(f"\nMatch count distribution:")
    print(matches["n_matches"].value_counts().sort("n_matches"))
    
    singleton_pct = (matches["n_matches"] == 0).mean() * 100
    print(f"\nSingleton rate: {singleton_pct:.1f}%")
    print(f"Avg matches per matched entity: {matches.filter(pl.col('n_matches') > 0)['n_matches'].mean():.2f}")
else:
    print("output/matching_results.tsv not found - run full inference first")

In [ ]:
# Analyze candidate_pairs.tsv
cand_path = "output/candidate_pairs.tsv"
if os.path.exists(cand_path):
    candidates = pl.read_csv(cand_path, separator="\t")
    candidates = candidates.with_columns(
        pl.col("candidate_entity_ids").str.split(",").alias("cand_list")
    ).with_columns(
        pl.when(pl.col("candidate_entity_ids") == "").then(0)
        .otherwise(pl.col("cand_list").list.lengths()).alias("n_cands")
    )
    
    print(f"Total S1 entities: {candidates.height:,}")
    print(f"Avg candidates per S1: {candidates['n_cands'].mean():.1f}")
    print(f"Median candidates: {candidates['n_cands'].median()}")
    print(f"Max candidates: {candidates['n_cands'].max()}")
    print(f"Entities with 0 candidates: {(candidates['n_cands'] == 0).sum():,} ({(candidates['n_cands'] == 0).mean()*100:.1f}%)")
    
    # Reduction ratio
    total_corpus = 10300000  # approximate
    reduction = total_corpus / candidates['n_cands'].mean()
    print(f"\nReduction ratio: ~{reduction:,.0f}x (vs all-pairs)")
else:
    print("output/candidate_pairs.tsv not found - run full inference first")

## 10. Create Final Submission Package

In [19]:
# Create submission zip
import zipfile
from datetime import datetime

team_name = "your_team_name"  # CHANGE THIS
zip_name = f"{team_name}_submission_{datetime.now().strftime('%Y%m%d')}.zip"

files_to_include = [
    ("output/matching_results.tsv", "output/matching_results.tsv"),
    ("output/candidate_pairs.tsv", "output/candidate_pairs.tsv"),
    ("code/business_entity_resolution", "code/business_entity_resolution"),
    ("Documentation_template.md", "Documentation_template.md"),
]

print(f"Creating {zip_name}...")
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for src, arc in files_to_include:
        src_path = os.path.join(BASE, src)
        if os.path.isfile(src_path):
            zf.write(src_path, arc)
            print(f"  Added file: {arc}")
        elif os.path.isdir(src_path):
            for root, dirs, files in os.walk(src_path):
                for f in files:
                    full = os.path.join(root, f)
                    rel = os.path.relpath(full, BASE)
                    zf.write(full, rel)
            print(f"  Added dir: {arc}")
        else:
            print(f"  WARNING: Missing {src}")

print(f"\nZip created: {zip_name}")
print(f"Size: {os.path.getsize(os.path.join(BASE, zip_name))/1e6:.1f} MB")

Creating your_team_name_submission_20260927.zip...


NameError: name 'BASE' is not defined

## 11. Monitoring & Debugging Utilities

In [20]:
# Disk usage monitor
import shutil

def check_disk(path="."):
    total, used, free = shutil.disk_usage(path)
    print(f"Disk: {used/1e9:.1f} GB used / {total/1e9:.1f} GB total ({free/1e9:.1f} GB free)")

def check_work_size():
    work_path = "work"
    if os.path.exists(work_path):
        total = sum(os.path.getsize(os.path.join(dp, f)) for dp, dn, fn in os.walk(work_path) for f in fn)
        print(f"Work dir: {total/1e9:.2f} GB")
        for root, dirs, files in os.walk(work_path):
            for f in files:
                fp = os.path.join(root, f)
                sz = os.path.getsize(fp) / 1e6
                if sz > 10:
                    print(f"  {os.path.relpath(fp, BASE)}: {sz:.1f} MB")

check_disk()
check_work_size()

Disk: 504.0 GB used / 512.0 GB total (8.0 GB free)
Work dir: 2.53 GB


NameError: name 'BASE' is not defined

In [21]:
# Memory monitor
import psutil

mem = psutil.virtual_memory()
print(f"RAM: {mem.used/1e9:.1f} / {mem.total/1e9:.1f} GB ({mem.percent:.1f}%)")
print(f"Available: {mem.available/1e9:.1f} GB")

swap = psutil.swap_memory()
print(f"Swap: {swap.used/1e9:.1f} / {swap.total/1e9:.1f} GB")

RAM: 16.3 / 17.0 GB (96.0%)
Available: 0.7 GB
Swap: 4.3 / 18.2 GB


In [22]:
# Quick blocking recall check (on training data)
# This estimates how many true matches are in the candidate set

from pipeline import build_corpus_and_index, open_corpus, process_batch
from norm_store import Store
from normalize import h64
import numpy as np

# Load small training slice
index, tokdf, s1_path, n_corpus = build_corpus_and_index(
    "train", "_full", workers=8, keep_corpus=True, limit=500000
)
corpus_path, corpus = open_corpus("train", "_full")
s1 = Store(s1_path)

# Check first 1000 S1 entities
ps, pd, sc, X, fb = process_batch(index, tokdf, corpus, s1, 0, 1000,
                                   top_k=100, key_budget=14, per_key_cap=80)

# Load GT for these entities
gt = {}
with open("dataset/train/train_ground_truth.tsv") as f:
    next(f)
    for line in f:
        s1_id, _, rest = line.rstrip("\n").partition("\t")
        if s1_id in s1.ids[:1000]:
            gt[s1_id] = [x for x in rest.split(",") if x]

# Check recall
corpus_id_hash = corpus.id_hash
found = 0
total_pos = 0
for i, s1_id in enumerate(s1.ids[:1000]):
    truth = gt.get(s1_id, [])
    total_pos += len(truth)
    if not truth:
        continue
    ths = np.fromiter((h64("id", x) for x in truth), dtype=np.uint64, count=len(truth))
    mask = ps == i
    cand_hashes = corpus_id_hash[pd[mask]]
    found += np.isin(ths, cand_hashes).sum()

print(f"Blocking recall on 1000 train entities: {found}/{total_pos} = {found/max(total_pos,1)*100:.1f}%")
print(f"Avg candidates per entity: {ps.size/1000:.1f}")

ModuleNotFoundError: No module named 'pipeline'

## Summary & Next Steps

### ✅ Completed:
- Environment verified
- EDA complete
- Smoke test passed
- Model inspected

### 🎯 To Reach 99th Percentile:
1. **Run full inference** (Cell 3) → get baseline leaderboard score
2. **Analyze errors** by country, name type, address pattern
3. **Run tuning experiments** (Cell 6) focusing on:
   - `top_k` 80-120 (smaller = better ranking)
   - `key_budget` 12-16 (more keys = better recall)
   - `per_key_cap` 60-100
4. **Retrain model** with optimized params (Cell 7)
5. **Add France-specific features** (Cell 8)
6. **Validate & submit** (Cells 4, 10)

### 📊 Key Metrics to Track:
- Blocking recall (matchable entities) > 99.5%
- Candidates per S1 entity < 80
- Holdout F₀.₅ > 0.82
- Inference time < 2 hours

Good luck! 🚀